In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

In [2]:
required_names = [
    "X_train_fit",
    "X_validation",
    "y_train_fit",
    "y_validation",
    "X_test",
    "y_test",
    "category_order",
]

for name in required_names:
    available = name in globals()
    print(f"{name}: {'available' if available else 'NOT available'}")
    if not available:
        print()
        continue

    obj = globals()[name]
    if hasattr(obj, "shape"):
        print("  shape:", obj.shape)
    elif hasattr(obj, "__len__"):
        print("  length:", len(obj))
    else:
        print("  value:", obj)

    if name in ("y_train_fit", "y_validation", "y_test"):
        print("  class counts:")
        display(obj.value_counts())
    print()

X_train_fit: NOT available

X_validation: NOT available

y_train_fit: NOT available

y_validation: NOT available

X_test: NOT available

y_test: NOT available

category_order: NOT available



# Day 5 — Data Preparation

This notebook is **self-contained**. It reloads NSL-KDD with the same column names, five-class `attack_category` map, feature lists, and 80/20 stratified split as Days 3–4.

KDDTest+ is loaded as `X_test` / `y_test` but is **not** used to split, fit, or choose models. No Random Forest, SMOTE, class weights, or hyperparameter tuning in this section.

In [3]:
from sklearn.model_selection import train_test_split

COLUMNS = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label",
    "difficulty",
]

attack_category_map = {
    "normal": "Normal",
    "back": "DoS",
    "land": "DoS",
    "neptune": "DoS",
    "pod": "DoS",
    "smurf": "DoS",
    "teardrop": "DoS",
    "apache2": "DoS",
    "udpstorm": "DoS",
    "processtable": "DoS",
    "mailbomb": "DoS",
    "worm": "DoS",
    "satan": "Probe",
    "ipsweep": "Probe",
    "nmap": "Probe",
    "portsweep": "Probe",
    "mscan": "Probe",
    "saint": "Probe",
    "guess_passwd": "R2L",
    "ftp_write": "R2L",
    "imap": "R2L",
    "phf": "R2L",
    "multihop": "R2L",
    "warezmaster": "R2L",
    "warezclient": "R2L",
    "spy": "R2L",
    "xlock": "R2L",
    "xsnoop": "R2L",
    "snmpguess": "R2L",
    "snmpgetattack": "R2L",
    "httptunnel": "R2L",
    "sendmail": "R2L",
    "named": "R2L",
    "buffer_overflow": "U2R",
    "loadmodule": "U2R",
    "rootkit": "U2R",
    "perl": "U2R",
    "sqlattack": "U2R",
    "xterm": "U2R",
    "ps": "U2R",
}

if "category_order" not in globals():
    category_order = ["Normal", "DoS", "Probe", "R2L", "U2R"]

feature_columns = [col for col in COLUMNS if col not in ["label", "difficulty"]]
categorical_features = ["protocol_type", "service", "flag"]
numerical_features = [col for col in feature_columns if col not in categorical_features]

train_df = pd.read_csv("../data/raw/KDDTrain+.txt", header=None, names=COLUMNS)
test_df = pd.read_csv("../data/raw/KDDTest+.txt", header=None, names=COLUMNS)

train_df["attack_category"] = train_df["label"].map(attack_category_map)
test_df["attack_category"] = test_df["label"].map(attack_category_map)

X_train = train_df[feature_columns]
y_train = train_df["attack_category"]
X_test = test_df[feature_columns]
y_test = test_df["attack_category"]

X_train_fit, X_validation, y_train_fit, y_validation = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=42,
)

print("1. X_train_fit shape:", X_train_fit.shape)
print("2. X_validation shape:", X_validation.shape)
print("3. y_train_fit shape:", y_train_fit.shape)
print("4. y_validation shape:", y_validation.shape)
print("5. y_train_fit class counts:")
display(y_train_fit.value_counts().reindex(category_order))
print("6. y_validation class counts:")
display(y_validation.value_counts().reindex(category_order))
print("7. X_test shape:", X_test.shape)
print("8. y_test shape:", y_test.shape)
print("9. category_order:", category_order)
print()
print("KDDTest+ is held out. No Random Forest, SMOTE, or class_weight in this section.")

1. X_train_fit shape: (100778, 41)
2. X_validation shape: (25195, 41)
3. y_train_fit shape: (100778,)
4. y_validation shape: (25195,)
5. y_train_fit class counts:


attack_category
Normal    53874
DoS       36741
Probe      9325
R2L         796
U2R          42
Name: count, dtype: int64

6. y_validation class counts:


attack_category
Normal    13469
DoS        9186
Probe      2331
R2L         199
U2R          10
Name: count, dtype: int64

7. X_test shape: (22544, 41)
8. y_test shape: (22544,)
9. category_order: ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

KDDTest+ is held out. No Random Forest, SMOTE, or class_weight in this section.


# Random Forest — Validation Baseline

Plain `RandomForestClassifier` (`n_estimators=300`, `random_state=42`, `n_jobs=-1`). **No** `class_weight`, **no** SMOTE.

Categorical strings (`protocol_type`, `service`, `flag`) are one-hot encoded; numeric columns use `StandardScaler`, matching the project preprocessor. That preprocessor is **fitted on `X_train_fit` only**, then applied to `X_validation`. KDDTest+ is not used.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rf_baseline_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features),
    ]
)

X_train_fit_processed = rf_baseline_preprocessor.fit_transform(X_train_fit)
X_validation_processed = rf_baseline_preprocessor.transform(X_validation)

rf_validation_baseline = RandomForestClassifier(
    random_state=42,
    n_estimators=300,
    n_jobs=-1,
)
rf_validation_baseline.fit(X_train_fit_processed, y_train_fit)
y_validation_pred_rf_baseline = rf_validation_baseline.predict(X_validation_processed)

rf_val_acc = accuracy_score(y_validation, y_validation_pred_rf_baseline)
rf_val_macro_p = precision_score(
    y_validation, y_validation_pred_rf_baseline, average="macro", labels=category_order, zero_division=0
)
rf_val_macro_r = recall_score(
    y_validation, y_validation_pred_rf_baseline, average="macro", labels=category_order, zero_division=0
)
rf_val_macro_f1 = f1_score(
    y_validation, y_validation_pred_rf_baseline, average="macro", labels=category_order, zero_division=0
)

print("Validation — Random Forest baseline (no class_weight, no SMOTE)")
print(f"Accuracy:        {rf_val_acc:.4f}")
print(f"Macro precision: {rf_val_macro_p:.4f}")
print(f"Macro recall:    {rf_val_macro_r:.4f}")
print(f"Macro F1:        {rf_val_macro_f1:.4f}")
print()
print(classification_report(
    y_validation,
    y_validation_pred_rf_baseline,
    labels=category_order,
    digits=4,
    zero_division=0,
))

rf_val_cm = confusion_matrix(
    y_validation, y_validation_pred_rf_baseline, labels=category_order
)
rf_val_cm_df = pd.DataFrame(
    rf_val_cm,
    index=["Actual " + c for c in category_order],
    columns=["Predicted " + c for c in category_order],
)
print("Confusion matrix (rows = actual, columns = predicted):")
display(rf_val_cm_df)

rf_val_prec, rf_val_rec, rf_val_f1s, _ = precision_recall_fscore_support(
    y_validation,
    y_validation_pred_rf_baseline,
    labels=category_order,
    zero_division=0,
)
rf_val_per_class = pd.DataFrame(
    {"precision": rf_val_prec, "recall": rf_val_rec, "f1": rf_val_f1s},
    index=category_order,
)
print("R2L — precision, recall, F1:")
display(rf_val_per_class.loc[["R2L"]])
print("U2R — precision, recall, F1:")
display(rf_val_per_class.loc[["U2R"]])

print()
print("Stored: rf_baseline_preprocessor, rf_validation_baseline, y_validation_pred_rf_baseline")
print("KDDTest+ remains untouched. No SMOTE, class_weight, or threshold tuning was used.")

Validation — Random Forest baseline (no class_weight, no SMOTE)
Accuracy:        0.9988
Macro precision: 0.9745
Macro recall:    0.9306
Macro F1:        0.9505

              precision    recall  f1-score   support

      Normal     0.9980    0.9997    0.9989     13469
         DoS     0.9998    0.9999    0.9998      9186
       Probe     0.9996    0.9936    0.9966      2331
         R2L     1.0000    0.9598    0.9795       199
         U2R     0.8750    0.7000    0.7778        10

    accuracy                         0.9988     25195
   macro avg     0.9745    0.9306    0.9505     25195
weighted avg     0.9988    0.9988    0.9988     25195

Confusion matrix (rows = actual, columns = predicted):


,Predicted Normal,Predicted DoS,Predicted Probe,Predicted R2L,Predicted U2R
Actual Normal,13465,2,1,0,1
Actual DoS,1,9185,0,0,0
Actual Probe,15,0,2316,0,0
Actual R2L,8,0,0,191,0
Actual U2R,3,0,0,0,7


R2L — precision, recall, F1:


,precision,recall,f1
R2L,1.0,0.959799,0.979487


U2R — precision, recall, F1:


,precision,recall,f1
U2R,0.875,0.7,0.777778



Stored: rf_baseline_preprocessor, rf_validation_baseline, y_validation_pred_rf_baseline
KDDTest+ remains untouched. No SMOTE, class_weight, or threshold tuning was used.


# Random Forest — Validation Sensitivity to Number of Trees

Compare `n_estimators` in `{100, 300, 500}` on the **validation** split only.

Reuse `rf_baseline_preprocessor` (already fitted on `X_train_fit`). Do **not** refit it. No `class_weight`, SMOTE, threshold tuning, or KDDTest+.

In [5]:
X_train_fit_for_trees = rf_baseline_preprocessor.transform(X_train_fit)
X_validation_for_trees = rf_baseline_preprocessor.transform(X_validation)

tree_counts = [100, 300, 500]
tree_rows = []

for n_trees in tree_counts:
    rf_k = RandomForestClassifier(
        random_state=42,
        n_estimators=n_trees,
        n_jobs=-1,
    )
    rf_k.fit(X_train_fit_for_trees, y_train_fit)
    y_val_k = rf_k.predict(X_validation_for_trees)

    acc = accuracy_score(y_validation, y_val_k)
    macro_p = precision_score(
        y_validation, y_val_k, average="macro", labels=category_order, zero_division=0
    )
    macro_r = recall_score(
        y_validation, y_val_k, average="macro", labels=category_order, zero_division=0
    )
    macro_f1 = f1_score(
        y_validation, y_val_k, average="macro", labels=category_order, zero_division=0
    )
    prec, rec, f1s, _ = precision_recall_fscore_support(
        y_validation, y_val_k, labels=category_order, zero_division=0
    )
    per_class = pd.DataFrame(
        {"precision": prec, "recall": rec, "f1": f1s}, index=category_order
    )
    tree_rows.append(
        {
            "n_estimators": n_trees,
            "Accuracy": acc,
            "Macro Precision": macro_p,
            "Macro Recall": macro_r,
            "Macro F1": macro_f1,
            "R2L Precision": per_class.loc["R2L", "precision"],
            "R2L Recall": per_class.loc["R2L", "recall"],
            "R2L F1": per_class.loc["R2L", "f1"],
            "U2R Precision": per_class.loc["U2R", "precision"],
            "U2R Recall": per_class.loc["U2R", "recall"],
            "U2R F1": per_class.loc["U2R", "f1"],
        }
    )

rf_tree_count_comparison = pd.DataFrame(tree_rows)
print("Random Forest validation comparison by n_estimators:")
display(rf_tree_count_comparison)

print()
print("KDDTest+ remains untouched. Tree-count comparison was performed on validation only.")

Random Forest validation comparison by n_estimators:


,n_estimators,Accuracy,Macro Precision,Macro Recall,Macro F1,R2L Precision,R2L Recall,R2L F1,U2R Precision,U2R Recall,U2R F1
0,100,0.998730,0.974384,0.930577,0.950452,1.0,0.959799,0.979487,0.875,0.7,0.777778
1,300,0.998770,0.974470,0.930592,0.950502,1.0,0.959799,0.979487,0.875,0.7,0.777778
2,500,0.998809,0.974485,0.931597,0.951031,1.0,0.964824,0.982097,0.875,0.7,0.777778



KDDTest+ remains untouched. Tree-count comparison was performed on validation only.


# Validation Comparison — Random Forest vs Logistic Regression Baselines

**Validation / model-selection only** — not a final KDDTest+ evaluation.

Logistic Regression rows use **existing Day 4 validation metrics** (variables if present in this kernel; otherwise the exact numbers already reported in Day 4). Those models are **not** retrained. The Random Forest row uses the **500-tree** validation row already stored in `rf_tree_count_comparison`.

In [6]:
def _row_from_metrics_dict(evaluation, metrics):
    pc = metrics["per_class"]
    return {
        "Evaluation": evaluation,
        "Accuracy": metrics["accuracy"],
        "Macro Precision": metrics["macro_precision"],
        "Macro Recall": metrics["macro_recall"],
        "Macro F1": metrics["macro_f1"],
        "R2L Precision": pc.loc["R2L", "precision"],
        "R2L Recall": pc.loc["R2L", "recall"],
        "R2L F1": pc.loc["R2L", "f1"],
        "U2R Precision": pc.loc["U2R", "precision"],
        "U2R Recall": pc.loc["U2R", "recall"],
        "U2R F1": pc.loc["U2R", "f1"],
    }


def _row_from_fallback(evaluation):
    row = dict(day4_validation_fallback[evaluation])
    row["Evaluation"] = evaluation
    return row


# Exact Day 4 validation numbers (used when those models are not in this kernel)
day4_validation_fallback = {
    "Class-weighted Logistic Regression": {
        "Accuracy": 0.9668,
        "Macro Precision": 0.6567,
        "Macro Recall": 0.9528,
        "Macro F1": 0.6985,
        "R2L Precision": 0.3091,
        "R2L Recall": 0.9397,
        "R2L F1": 0.4652,
        "U2R Precision": 0.0577,
        "U2R Recall": 0.9000,
        "U2R F1": 0.1084,
    },
    "Logistic Regression — no SMOTE": {
        "Accuracy": 0.9889,
        "Macro Precision": 0.9420,
        "Macro Recall": 0.8216,
        "Macro F1": 0.8560,
        "R2L Precision": 0.7438,
        "R2L Recall": 0.7588,
        "R2L F1": 0.7512,
        "U2R Precision": 1.0000,
        "U2R Recall": 0.4000,
        "U2R F1": 0.5714,
    },
    "Logistic Regression — SMOTE": {
        "Accuracy": 0.9698,
        "Macro Precision": 0.6616,
        "Macro Recall": 0.9313,
        "Macro F1": 0.7028,
        "R2L Precision": 0.3292,
        "R2L Recall": 0.9246,
        "R2L F1": 0.4855,
        "U2R Precision": 0.0544,
        "U2R Recall": 0.8000,
        "U2R F1": 0.1019,
    },
}

rows = []
sources = []

# Logistic Regression: reuse Day 4 validation results only. Do not retrain.
if "validation_comparison_df" in globals():
    lr_from_day4 = validation_comparison_df.copy()
    for evaluation in (
        "Class-weighted Logistic Regression",
        "Logistic Regression — no SMOTE",
        "Logistic Regression — SMOTE",
    ):
        match = lr_from_day4.loc[lr_from_day4["Evaluation"] == evaluation]
        if len(match) == 1:
            rows.append(match.iloc[0].to_dict())
            sources.append(f"{evaluation}: validation_comparison_df (Day 4 validation)")
        else:
            rows.append(_row_from_fallback(evaluation))
            sources.append(f"{evaluation}: Day 4 reported validation metrics (not retrained)")
else:
    rows.append(_row_from_fallback("Class-weighted Logistic Regression"))
    sources.append(
        "Class-weighted LR: Day 4 reported validation metrics (not retrained)"
    )

    if "no_smote_metrics" in globals() and "per_class" in no_smote_metrics:
        rows.append(
            _row_from_metrics_dict("Logistic Regression — no SMOTE", no_smote_metrics)
        )
        sources.append("No SMOTE LR: kernel variable no_smote_metrics (validation)")
    else:
        rows.append(_row_from_fallback("Logistic Regression — no SMOTE"))
        sources.append("No SMOTE LR: Day 4 reported validation metrics (not retrained)")

    if "smote_metrics" in globals() and "per_class" in smote_metrics:
        rows.append(_row_from_metrics_dict("Logistic Regression — SMOTE", smote_metrics))
        sources.append("SMOTE LR: kernel variable smote_metrics (validation)")
    else:
        rows.append(_row_from_fallback("Logistic Regression — SMOTE"))
        sources.append("SMOTE LR: Day 4 reported validation metrics (not retrained)")

if "rf_tree_count_comparison" not in globals():
    print("STOPPED. rf_tree_count_comparison is not in this kernel.")
    print("Run the n_estimators validation comparison first.")
    print("Logistic Regression was not retrained. KDDTest+ was not used.")
else:
    rf500 = rf_tree_count_comparison.loc[
        rf_tree_count_comparison["n_estimators"] == 500
    ].iloc[0]
    rows.append(
        {
            "Evaluation": "Random Forest — 500 trees",
            "Accuracy": rf500["Accuracy"],
            "Macro Precision": rf500["Macro Precision"],
            "Macro Recall": rf500["Macro Recall"],
            "Macro F1": rf500["Macro F1"],
            "R2L Precision": rf500["R2L Precision"],
            "R2L Recall": rf500["R2L Recall"],
            "R2L F1": rf500["R2L F1"],
            "U2R Precision": rf500["U2R Precision"],
            "U2R Recall": rf500["U2R Recall"],
            "U2R F1": rf500["U2R F1"],
        }
    )
    sources.append("Random Forest 500: rf_tree_count_comparison (validation)")

    metric_cols = [
        "Evaluation",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "R2L Precision",
        "R2L Recall",
        "R2L F1",
        "U2R Precision",
        "U2R Recall",
        "U2R F1",
    ]
    day5_validation_model_comparison = pd.DataFrame(rows)[metric_cols]

    print("Validation / model-selection comparison (not final KDDTest+ evaluation)")
    print()
    for line in sources:
        print(" ", line)
    print()
    display(day5_validation_model_comparison)

    print()
    print("KDDTest+ remains untouched. No final test evaluation has been performed.")


Validation / model-selection comparison (not final KDDTest+ evaluation)

  Class-weighted LR: Day 4 reported validation metrics (not retrained)
  No SMOTE LR: Day 4 reported validation metrics (not retrained)
  SMOTE LR: Day 4 reported validation metrics (not retrained)
  Random Forest 500: rf_tree_count_comparison (validation)



,Evaluation,Accuracy,Macro Precision,Macro Recall,Macro F1,R2L Precision,R2L Recall,R2L F1,U2R Precision,U2R Recall,U2R F1
0,Class-weighted Logistic Regression,0.966800,0.656700,0.952800,0.698500,0.3091,0.939700,0.465200,0.0577,0.9,0.108400
1,Logistic Regression — no SMOTE,0.988900,0.942000,0.821600,0.856000,0.7438,0.758800,0.751200,1.0000,0.4,0.571400
2,Logistic Regression — SMOTE,0.969800,0.661600,0.931300,0.702800,0.3292,0.924600,0.485500,0.0544,0.8,0.101900
3,Random Forest — 500 trees,0.998809,0.974485,0.931597,0.951031,1.0000,0.964824,0.982097,0.8750,0.7,0.777778



KDDTest+ remains untouched. No final test evaluation has been performed.


# Final Model Lock — Random Forest 500 Trees

Based **strictly** on the validation / model-selection comparison already completed. This cell does **not** search hyperparameters, retrain Logistic Regression, or evaluate KDDTest+.

**Locked configuration**

- Model: `RandomForestClassifier`
- `n_estimators=500`
- `random_state=42`
- `n_jobs=-1`
- `class_weight=None`
- No SMOTE
- No threshold tuning

In [7]:
def _is_locked_rf500(model):
    return (
        isinstance(model, RandomForestClassifier)
        and getattr(model, "n_estimators", None) == 500
        and getattr(model, "random_state", None) == 42
        and getattr(model, "class_weight", None) is None
    )


missing = [
    name
    for name in ("rf_baseline_preprocessor", "X_train_fit", "y_train_fit")
    if name not in globals()
]
if missing:
    print("STOPPED. Missing:", ", ".join(missing))
    print("Run the Day 5 data-preparation and RF baseline cells first.")
    print("KDDTest+ was not used.")
else:
    model_source = None
    if "rf_final_model" in globals() and _is_locked_rf500(rf_final_model):
        model_source = "existing rf_final_model"
    elif "rf_k" in globals() and _is_locked_rf500(rf_k):
        rf_final_model = rf_k
        model_source = "existing 500-tree model from the tree-count loop (rf_k)"
    else:
        X_train_fit_locked = rf_baseline_preprocessor.transform(X_train_fit)
        rf_final_model = RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            n_jobs=-1,
            class_weight=None,
        )
        rf_final_model.fit(X_train_fit_locked, y_train_fit)
        model_source = (
            "trained on X_train_fit / y_train_fit using already-fitted "
            "rf_baseline_preprocessor (not refit)"
        )

    val_macro_f1 = val_r2l_f1 = val_u2r_f1 = None
    metric_source = None
    if "day5_validation_model_comparison" in globals():
        match = day5_validation_model_comparison.loc[
            day5_validation_model_comparison["Evaluation"] == "Random Forest — 500 trees"
        ]
        if len(match) == 1:
            val_macro_f1 = match.iloc[0]["Macro F1"]
            val_r2l_f1 = match.iloc[0]["R2L F1"]
            val_u2r_f1 = match.iloc[0]["U2R F1"]
            metric_source = "day5_validation_model_comparison (validation)"
    if val_macro_f1 is None and "rf_tree_count_comparison" in globals():
        match = rf_tree_count_comparison.loc[
            rf_tree_count_comparison["n_estimators"] == 500
        ]
        if len(match) == 1:
            val_macro_f1 = match.iloc[0]["Macro F1"]
            val_r2l_f1 = match.iloc[0]["R2L F1"]
            val_u2r_f1 = match.iloc[0]["U2R F1"]
            metric_source = "rf_tree_count_comparison (validation)"

    print("Final model lock (validation / model-selection only; not KDDTest+ evaluation)")
    print()
    print("Selected model:            RandomForestClassifier")
    print("Number of trees:           500")
    print("class_weight:              None")
    print("SMOTE status:              No SMOTE")
    print("threshold tuning status:   No threshold tuning")
    if val_macro_f1 is None:
        print("validation Macro F1:       unavailable — run the validation comparison first")
        print("validation R2L F1:         unavailable")
        print("validation U2R F1:         unavailable")
    else:
        print(f"validation Macro F1:       {val_macro_f1:.4f}")
        print(f"validation R2L F1:         {val_r2l_f1:.4f}")
        print(f"validation U2R F1:         {val_u2r_f1:.4f}")
        print(f"Validation metrics source: {metric_source}")
    print(f"Locked model object:       rf_final_model ({model_source})")
    print()
    print("Model selection is complete.")
    print("Random Forest 500 trees is locked as the final model.")
    print("KDDTest+ has not been used for model selection.")


Final model lock (validation / model-selection only; not KDDTest+ evaluation)

Selected model:            RandomForestClassifier
Number of trees:           500
class_weight:              None
SMOTE status:              No SMOTE
threshold tuning status:   No threshold tuning
validation Macro F1:       0.9510
validation R2L F1:         0.9821
validation U2R F1:         0.7778
Validation metrics source: day5_validation_model_comparison (validation)
Locked model object:       rf_final_model (existing 500-tree model from the tree-count loop (rf_k))

Model selection is complete.
Random Forest 500 trees is locked as the final model.
KDDTest+ has not been used for model selection.


# Day 5 — Final KDDTest+ Evaluation of the Locked Random Forest

Model selection is **complete**. The locked object is the existing `rf_final_model` (500-tree `RandomForestClassifier`). This section evaluates that model on **KDDTest+ only**.

No retraining, no new `RandomForestClassifier`, no `class_weight`, no SMOTE, no threshold tuning, and no preprocessor `fit` on `X_test`. KDDTest+ is not used to change or re-select the model.

In [8]:
required_final_eval = [
    "rf_final_model",
    "X_test",
    "y_test",
    "category_order",
    "rf_baseline_preprocessor",
]
missing_final_eval = [name for name in required_final_eval if name not in globals()]

if missing_final_eval:
    print("STOPPED. Missing variables:")
    for name in missing_final_eval:
        print(f"  {name}")
    print("Do not retrain or reconstruct anything to compensate.")
else:
    X_test_processed_rf = rf_baseline_preprocessor.transform(X_test)
    y_test_pred_rf = rf_final_model.predict(X_test_processed_rf)

    rf_test_accuracy = accuracy_score(y_test, y_test_pred_rf)
    rf_test_macro_precision = precision_score(
        y_test, y_test_pred_rf, average="macro", labels=category_order, zero_division=0
    )
    rf_test_macro_recall = recall_score(
        y_test, y_test_pred_rf, average="macro", labels=category_order, zero_division=0
    )
    rf_test_macro_f1 = f1_score(
        y_test, y_test_pred_rf, average="macro", labels=category_order, zero_division=0
    )
    rf_test_weighted_precision = precision_score(
        y_test, y_test_pred_rf, average="weighted", labels=category_order, zero_division=0
    )
    rf_test_weighted_recall = recall_score(
        y_test, y_test_pred_rf, average="weighted", labels=category_order, zero_division=0
    )
    rf_test_weighted_f1 = f1_score(
        y_test, y_test_pred_rf, average="weighted", labels=category_order, zero_division=0
    )

    rf_kddtest_metrics = {
        "accuracy": rf_test_accuracy,
        "macro_precision": rf_test_macro_precision,
        "macro_recall": rf_test_macro_recall,
        "macro_f1": rf_test_macro_f1,
        "weighted_precision": rf_test_weighted_precision,
        "weighted_recall": rf_test_weighted_recall,
        "weighted_f1": rf_test_weighted_f1,
    }

    print("A. Overall metrics — KDDTest+ (locked Random Forest, 500 trees)")
    print(f"Accuracy:            {rf_test_accuracy:.4f}")
    print(f"Macro precision:     {rf_test_macro_precision:.4f}")
    print(f"Macro recall:        {rf_test_macro_recall:.4f}")
    print(f"Macro F1:            {rf_test_macro_f1:.4f}")
    print(f"Weighted precision:  {rf_test_weighted_precision:.4f}")
    print(f"Weighted recall:     {rf_test_weighted_recall:.4f}")
    print(f"Weighted F1:         {rf_test_weighted_f1:.4f}")
    print()

    print("B. Classification report")
    print(
        classification_report(
            y_test,
            y_test_pred_rf,
            labels=category_order,
            digits=4,
            zero_division=0,
        )
    )

    rf_test_cm = confusion_matrix(y_test, y_test_pred_rf, labels=category_order)
    rf_test_cm_df = pd.DataFrame(
        rf_test_cm,
        index=["Actual " + c for c in category_order],
        columns=["Predicted " + c for c in category_order],
    )
    print("C. Confusion matrix (rows = actual, columns = predicted)")
    display(rf_test_cm_df)
    print()

    prec, rec, f1s, _ = precision_recall_fscore_support(
        y_test, y_test_pred_rf, labels=category_order, zero_division=0
    )
    rf_kddtest_per_class = pd.DataFrame(
        {"precision": prec, "recall": rec, "f1": f1s},
        index=category_order,
    )
    print("D. Per-class metrics")
    display(rf_kddtest_per_class)
    print()

    class_index = {label: i for i, label in enumerate(category_order)}

    def class_error_counts(label):
        i = class_index[label]
        tp = int(rf_test_cm[i, i])
        fp = int(rf_test_cm[:, i].sum() - tp)
        fn = int(rf_test_cm[i, :].sum() - tp)
        return tp, fp, fn

    r2l_tp, r2l_fp, r2l_fn = class_error_counts("R2L")
    print("E. R2L metrics")
    print(f"R2L precision:        {rf_kddtest_per_class.loc['R2L', 'precision']:.4f}")
    print(f"R2L recall:           {rf_kddtest_per_class.loc['R2L', 'recall']:.4f}")
    print(f"R2L F1:               {rf_kddtest_per_class.loc['R2L', 'f1']:.4f}")
    print(f"R2L true positives:   {r2l_tp}")
    print(f"R2L false positives:  {r2l_fp}")
    print(f"R2L false negatives:  {r2l_fn}")
    print()

    u2r_tp, u2r_fp, u2r_fn = class_error_counts("U2R")
    print("F. U2R metrics")
    print(f"U2R precision:        {rf_kddtest_per_class.loc['U2R', 'precision']:.4f}")
    print(f"U2R recall:           {rf_kddtest_per_class.loc['U2R', 'recall']:.4f}")
    print(f"U2R F1:               {rf_kddtest_per_class.loc['U2R', 'f1']:.4f}")
    print(f"U2R true positives:   {u2r_tp}")
    print(f"U2R false positives:  {u2r_fp}")
    print(f"U2R false negatives:  {u2r_fn}")
    print()

    print("Stored: y_test_pred_rf, rf_kddtest_metrics, rf_kddtest_per_class")
    print()
    print("KDDTest+ was used for final evaluation only.")
    print("No model selection was performed using KDDTest+.")
    print("No retraining was performed.")
    print("No SMOTE was used.")
    print("No class_weight was used.")
    print("No threshold tuning was performed.")
    print("Locked final model: Random Forest, 500 trees.")


A. Overall metrics — KDDTest+ (locked Random Forest, 500 trees)
Accuracy:            0.7447
Macro precision:     0.8198
Macro recall:        0.4896
Macro F1:            0.5061
Weighted precision:  0.8133
Weighted recall:     0.7447
Weighted F1:         0.7037

B. Classification report
              precision    recall  f1-score   support

      Normal     0.6424    0.9738    0.7741      9711
         DoS     0.9613    0.7701    0.8552      7460
       Probe     0.8500    0.5969    0.7013      2421
         R2L     0.9786    0.0475    0.0906      2885
         U2R     0.6667    0.0597    0.1096        67

    accuracy                         0.7447     22544
   macro avg     0.8198    0.4896    0.5061     22544
weighted avg     0.8133    0.7447    0.7037     22544

C. Confusion matrix (rows = actual, columns = predicted)


,Predicted Normal,Predicted DoS,Predicted Probe,Predicted R2L,Predicted U2R
Actual Normal,9457,67,186,0,1
Actual DoS,1649,5745,66,0,0
Actual Probe,812,164,1445,0,0
Actual R2L,2744,0,3,137,1
Actual U2R,60,0,0,3,4



D. Per-class metrics


,precision,recall,f1
Normal,0.642372,0.973844,0.774117
DoS,0.961345,0.770107,0.855165
Probe,0.850000,0.596861,0.701286
R2L,0.978571,0.047487,0.090579
U2R,0.666667,0.059701,0.109589



E. R2L metrics
R2L precision:        0.9786
R2L recall:           0.0475
R2L F1:               0.0906
R2L true positives:   137
R2L false positives:  3
R2L false negatives:  2748

F. U2R metrics
U2R precision:        0.6667
U2R recall:           0.0597
U2R F1:               0.1096
U2R true positives:   4
U2R false positives:  2
U2R false negatives:  63

Stored: y_test_pred_rf, rf_kddtest_metrics, rf_kddtest_per_class

KDDTest+ was used for final evaluation only.
No model selection was performed using KDDTest+.
No retraining was performed.
No SMOTE was used.
No class_weight was used.
No threshold tuning was performed.
Locked final model: Random Forest, 500 trees.


# Day 5 — Final Model Comparison and Conclusion

**Reporting / analysis only.** No retraining, no model changes, no hyperparameter or threshold tuning, no SMOTE, and no new KDDTest+ scoring.

Day 4 Logistic Regression KDDTest+ numbers are the already-reported final scores. The Random Forest row uses stored `rf_kddtest_metrics` and `rf_kddtest_per_class`.

The locked experiment model remains **Random Forest, 500 trees**, selected on **validation only**. These KDDTest+ numbers do **not** change that lock.

In [9]:
required_compare = ["rf_kddtest_metrics", "rf_kddtest_per_class"]
missing_compare = [name for name in required_compare if name not in globals()]

if missing_compare:
    print("STOPPED. Missing variables:")
    for name in missing_compare:
        print(f"  {name}")
    print("Run the final KDDTest+ evaluation first. Do not retrain or rescore to compensate.")
else:
    day4_kddtest_reported = {
        "Logistic Regression — no SMOTE": {
            "Accuracy": 0.7610,
            "Macro Precision": 0.7746,
            "Macro Recall": 0.5637,
            "Macro F1": 0.5806,
            "R2L Precision": 0.6282,
            "R2L Recall": 0.0170,
            "R2L F1": 0.0331,
            "U2R Precision": 0.7826,
            "U2R Recall": 0.2687,
            "U2R F1": 0.4000,
        },
        "Logistic Regression — SMOTE": {
            "Accuracy": 0.7823,
            "Macro Precision": 0.6734,
            "Macro Recall": 0.6459,
            "Macro F1": 0.5863,
            "R2L Precision": 0.8267,
            "R2L Recall": 0.1886,
            "R2L F1": 0.3071,
            "U2R Precision": 0.0688,
            "U2R Recall": 0.4776,
            "U2R F1": 0.1203,
        },
    }

    def row_from_day4_test(evaluation, metrics_dict):
        pc = metrics_dict["per_class"]
        return {
            "Evaluation": evaluation,
            "Accuracy": metrics_dict["accuracy"],
            "Macro Precision": metrics_dict["macro_precision"],
            "Macro Recall": metrics_dict["macro_recall"],
            "Macro F1": metrics_dict["macro_f1"],
            "R2L Precision": pc.loc["R2L", "precision"],
            "R2L Recall": pc.loc["R2L", "recall"],
            "R2L F1": pc.loc["R2L", "f1"],
            "U2R Precision": pc.loc["U2R", "precision"],
            "U2R Recall": pc.loc["U2R", "recall"],
            "U2R F1": pc.loc["U2R", "f1"],
        }

    if "no_smote_test" in globals() and "per_class" in no_smote_test:
        lr_no_smote_row = row_from_day4_test(
            "Logistic Regression — no SMOTE", no_smote_test
        )
    else:
        lr_no_smote_row = {"Evaluation": "Logistic Regression — no SMOTE"}
        lr_no_smote_row.update(day4_kddtest_reported["Logistic Regression — no SMOTE"])

    if "smote_test" in globals() and "per_class" in smote_test:
        lr_smote_row = row_from_day4_test("Logistic Regression — SMOTE", smote_test)
    else:
        lr_smote_row = {"Evaluation": "Logistic Regression — SMOTE"}
        lr_smote_row.update(day4_kddtest_reported["Logistic Regression — SMOTE"])

    rf_row = {
        "Evaluation": "Random Forest — 500 trees",
        "Accuracy": rf_kddtest_metrics["accuracy"],
        "Macro Precision": rf_kddtest_metrics["macro_precision"],
        "Macro Recall": rf_kddtest_metrics["macro_recall"],
        "Macro F1": rf_kddtest_metrics["macro_f1"],
        "R2L Precision": rf_kddtest_per_class.loc["R2L", "precision"],
        "R2L Recall": rf_kddtest_per_class.loc["R2L", "recall"],
        "R2L F1": rf_kddtest_per_class.loc["R2L", "f1"],
        "U2R Precision": rf_kddtest_per_class.loc["U2R", "precision"],
        "U2R Recall": rf_kddtest_per_class.loc["U2R", "recall"],
        "U2R F1": rf_kddtest_per_class.loc["U2R", "f1"],
    }

    final_kddtest_comparison = pd.DataFrame(
        [lr_no_smote_row, lr_smote_row, rf_row]
    )
    display_cols = [
        "Evaluation",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "R2L Precision",
        "R2L Recall",
        "R2L F1",
        "U2R Precision",
        "U2R Recall",
        "U2R F1",
    ]
    final_kddtest_comparison = final_kddtest_comparison[display_cols]

    print("KDDTest+ comparison (reporting only; locked model unchanged)")
    display(final_kddtest_comparison.round(4))
    print()

    metric_winners = [
        "Accuracy",
        "Macro F1",
        "R2L Recall",
        "R2L F1",
        "U2R Recall",
        "U2R F1",
    ]
    print("Highest KDDTest+ score by metric (reporting only; does not change the lock):")
    for metric in metric_winners:
        winner_idx = final_kddtest_comparison[metric].idxmax()
        winner = final_kddtest_comparison.loc[winner_idx, "Evaluation"]
        value = final_kddtest_comparison.loc[winner_idx, metric]
        print(f"  {metric}: {winner} ({value:.4f})")
    print()

    rf_test_macro_f1 = rf_kddtest_metrics["macro_f1"]
    rf_test_r2l_p = rf_kddtest_per_class.loc["R2L", "precision"]
    rf_test_r2l_r = rf_kddtest_per_class.loc["R2L", "recall"]
    rf_test_r2l_f1 = rf_kddtest_per_class.loc["R2L", "f1"]
    rf_test_u2r_p = rf_kddtest_per_class.loc["U2R", "precision"]
    rf_test_u2r_r = rf_kddtest_per_class.loc["U2R", "recall"]
    rf_test_u2r_f1 = rf_kddtest_per_class.loc["U2R", "f1"]
    lr_smote_macro_f1 = lr_smote_row["Macro F1"]
    lr_nonsmote_macro_f1 = lr_no_smote_row["Macro F1"]
    lr_smote_r2l_r = lr_smote_row["R2L Recall"]
    lr_nonsmote_r2l_r = lr_no_smote_row["R2L Recall"]
    lr_smote_u2r_r = lr_smote_row["U2R Recall"]
    lr_nonsmote_u2r_f1 = lr_no_smote_row["U2R F1"]

    print("Interpretation (factual; no new experiments)")
    print()
    print(
        "1. Random Forest achieved the strongest validation Macro F1 (0.9510) "
        "and was therefore selected before KDDTest+ evaluation."
    )
    print(
        "2. On KDDTest+, Random Forest's performance dropped substantially: "
        f"validation Macro F1 = 0.9510; KDDTest+ Macro F1 = {rf_test_macro_f1:.4f}."
    )
    print(
        "3. That drop is a substantial validation-to-test generalization gap: "
        "the validation split is drawn from KDDTrain+, while KDDTest+ includes "
        "a different attack mix and is harder, especially for rare classes."
    )
    print(
        "4. R2L and U2R are the main weaknesses of the Random Forest on KDDTest+ "
        f"(R2L precision {rf_test_r2l_p:.4f}, recall {rf_test_r2l_r:.4f}, "
        f"F1 {rf_test_r2l_f1:.4f}; U2R precision {rf_test_u2r_p:.4f}, "
        f"recall {rf_test_u2r_r:.4f}, F1 {rf_test_u2r_f1:.4f})."
    )
    print(
        "5. Compared with Day 4 Logistic Regression on the same KDDTest+ scores, "
        f"SMOTE LR Macro F1 is {lr_smote_macro_f1:.4f} and no-SMOTE LR Macro F1 is "
        f"{lr_nonsmote_macro_f1:.4f}. SMOTE LR has higher R2L recall "
        f"({lr_smote_r2l_r:.4f} vs RF {rf_test_r2l_r:.4f} and no-SMOTE LR "
        f"{lr_nonsmote_r2l_r:.4f}). No-SMOTE LR has the highest U2R F1 in this table "
        f"({lr_nonsmote_u2r_f1:.4f}); SMOTE LR has the highest U2R recall "
        f"({lr_smote_u2r_r:.4f}). These are observed KDDTest+ differences, not a reason "
        "to replace the validation-locked model."
    )
    print(
        "6. Random Forest is not claimed to be the best KDDTest+ model because it "
        "won validation. Validation selected the experiment's locked model; "
        "KDDTest+ only reports how that locked model generalized."
    )
    print(
        "7. No additional experimentation is performed from these KDDTest+ results."
    )
    print()
    print("Day 5 model selection: Random Forest 500 trees, selected using validation only.")
    print("KDDTest+ evaluation: completed once after model lock.")
    print("No post-test tuning was performed.")
    print("KDDTest+ was not used to select or modify the model.")


KDDTest+ comparison (reporting only; locked model unchanged)


,Evaluation,Accuracy,Macro Precision,Macro Recall,Macro F1,R2L Precision,R2L Recall,R2L F1,U2R Precision,U2R Recall,U2R F1
0,Logistic Regression — no SMOTE,0.7610,0.7746,0.5637,0.5806,0.6282,0.0170,0.0331,0.7826,0.2687,0.4000
1,Logistic Regression — SMOTE,0.7823,0.6734,0.6459,0.5863,0.8267,0.1886,0.3071,0.0688,0.4776,0.1203
2,Random Forest — 500 trees,0.7447,0.8198,0.4896,0.5061,0.9786,0.0475,0.0906,0.6667,0.0597,0.1096



Highest KDDTest+ score by metric (reporting only; does not change the lock):
  Accuracy: Logistic Regression — SMOTE (0.7823)
  Macro F1: Logistic Regression — SMOTE (0.5863)
  R2L Recall: Logistic Regression — SMOTE (0.1886)
  R2L F1: Logistic Regression — SMOTE (0.3071)
  U2R Recall: Logistic Regression — SMOTE (0.4776)
  U2R F1: Logistic Regression — no SMOTE (0.4000)

Interpretation (factual; no new experiments)

1. Random Forest achieved the strongest validation Macro F1 (0.9510) and was therefore selected before KDDTest+ evaluation.
2. On KDDTest+, Random Forest's performance dropped substantially: validation Macro F1 = 0.9510; KDDTest+ Macro F1 = 0.5061.
3. That drop is a substantial validation-to-test generalization gap: the validation split is drawn from KDDTrain+, while KDDTest+ includes a different attack mix and is harder, especially for rare classes.
4. R2L and U2R are the main weaknesses of the Random Forest on KDDTest+ (R2L precision 0.9786, recall 0.0475, F1 0.0906; U2

# Day 5 — Random Forest Baseline, Model Selection, and Final Evaluation

## 1. Objective

Day 5 evaluated Random Forest as an alternative to the Day 4 Logistic Regression approaches for five-class network traffic anomaly classification: **Normal**, **DoS**, **Probe**, **R2L**, and **U2R**.

## 2. Validation experiment

KDDTrain+ was split into training/fitting and validation subsets using stratification. Random Forest was evaluated **without SMOTE** and **without `class_weight`**. **KDDTest+ was kept completely untouched during model selection.**

`n_estimators` values **100**, **300**, and **500** were compared on **validation only**. **500 trees** was selected because it achieved the highest validation Macro F1.

**Random Forest — 500 trees validation**

- Accuracy: 0.998809
- Macro Precision: 0.974485
- Macro Recall: 0.931597
- Macro F1: 0.951031
- R2L Precision: 1.0000
- R2L Recall: 0.964824
- R2L F1: 0.982097
- U2R Precision: 0.8750
- U2R Recall: 0.7000
- U2R F1: 0.777778

## 3. Final model lock

The locked configuration:

- Model: `RandomForestClassifier`
- `n_estimators`: 500
- `class_weight`: None
- SMOTE: No
- Threshold tuning: No
- Model selection source: KDDTrain+ validation only
- Model object: `rf_final_model`

This model was **locked before** KDDTest+ evaluation.

## 4. Final KDDTest+ evaluation

Locked Random Forest (500 trees) on KDDTest+:

- Accuracy: 0.7447
- Macro Precision: 0.8198
- Macro Recall: 0.4896
- Macro F1: 0.5061
- Weighted Precision: 0.8133
- Weighted Recall: 0.7447
- Weighted F1: 0.7037

**R2L**

- Precision: 0.9786
- Recall: 0.0475
- F1: 0.0906

**U2R**

- Precision: 0.6667
- Recall: 0.0597
- F1: 0.1096

## 5. Validation-to-test generalization

Random Forest had validation Macro F1 = **0.9510** but KDDTest+ Macro F1 = **0.5061**. That is a **substantial validation-to-test generalization gap**.

The validation split comes from KDDTrain+, whereas KDDTest+ contains a different and more challenging distribution, particularly for rare attack categories.

## 6. Comparison with Day 4 Logistic Regression

Final KDDTest+ comparison:

| Model | Accuracy | Macro F1 | R2L Recall | R2L F1 | U2R Recall | U2R F1 |
|---|---:|---:|---:|---:|---:|---:|
| Logistic Regression — no SMOTE | 0.7610 | 0.5806 | 0.0170 | 0.0331 | 0.2687 | 0.4000 |
| Logistic Regression — SMOTE | 0.7823 | 0.5863 | 0.1886 | 0.3071 | 0.4776 | 0.1203 |
| Random Forest — 500 trees | 0.7447 | 0.5061 | 0.0475 | 0.0906 | 0.0597 | 0.1096 |

The Logistic Regression models achieved higher KDDTest+ Macro F1 than the locked Random Forest. That does **not** change the Day 5 model lock, because KDDTest+ was reserved for final evaluation.

## 7. Important methodological conclusion

The Random Forest was selected using validation performance only. KDDTest+ was used once for final evaluation after model lock and was not used to tune, modify, or replace the selected model.

- No post-test tuning was performed.
- No additional experiments were performed after seeing KDDTest+.
- The KDDTest+ results are therefore a genuine held-out evaluation of the locked model.

## 8. Key finding

Random Forest produced extremely strong validation performance, but its KDDTest+ performance dropped substantially, particularly for R2L and U2R. This demonstrates that strong validation performance on KDDTrain+ does not necessarily imply strong cross-distribution generalization on KDDTest+. Random Forest is not claimed to be the best model on KDDTest+, and the locked model is not changed.